# OFI feature predictiveness — IC vs forward returns

**Question:** does OFI computed from top-of-book BBO data predict future mid-price moves?

**Method:**
1. Load a canon file's BBO records into a DataFrame.
2. Stream the BBO through `bpt_features.OFICalculator` to get a per-tick OFI value.
3. For each row, compute the forward log-return at horizons 100ms / 1s / 10s.
4. Spearman rank correlation between OFI and forward return = the Information Coefficient.

**Interpretation:**
- `|IC| > 0.05` → strong signal, almost certainly tradeable
- `|IC| 0.02-0.05` → meaningful, model it and combine with other features
- `|IC| < 0.01` → noise, look elsewhere

Sample size matters — IC of 0.01 over 10M ticks is still statistically significant.

## Setup — imports + path config

`bpt_canon` is pure-Python; `bpt_features` has a compiled `_core.so` that needs to be on the path. Adjust `REPO_ROOT` if running from outside the repo. After `bazel build //bpt-features/python/bpt_features:_core`, the `.so` lives at `bazel-bin/bpt-features/python/bpt_features/_core.so`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('/home/jseow/code/bpt-core')

# Pure-Python canon reader
sys.path.insert(0, str(REPO_ROOT / 'bpt-canon' / 'python'))
# Python wrappers for the compiled feature library
sys.path.insert(0, str(REPO_ROOT / 'bpt-features' / 'python'))
# Bazel-built _core.so lives next to the .py wrappers post-build
sys.path.insert(0, str(REPO_ROOT / 'bazel-bin' / 'bpt-features' / 'python'))

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import bpt_canon as bc
import bpt_features as bf

print(f'canon module: {bc.__file__}')
print(f'features module: {bf.__file__}')

## Config

Point `CANON_PATH` at a canon file you have. If none exist, generate one e.g. via `bpt-canon-ingest-okx-l2` over a day's OKX archive — see `bpt-canon/README.md`.

In [ ]:
CANON_PATH = Path('/tmp/bpt_canon/hl-2026-04-25.canon')  # adjust to your file

# Horizons to evaluate, in nanoseconds
HORIZONS_NS = {
    '100ms':   100_000_000,
    '1s':    1_000_000_000,
    '10s':  10_000_000_000,
}

# OFI calculator config — same as AS uses today
OFI_CFG = bf.OFICalculator.Config()
OFI_CFG.max_levels = 1            # BBO data only — one level
OFI_CFG.window_ns = 1_000_000_000 # 1-second rolling window

assert CANON_PATH.exists(), f'canon file not found: {CANON_PATH}'

## Load BBO records

If the file covers multiple instruments, filter to one — IC is per-instrument.

In [ ]:
bbo = bc.read_bbos(CANON_PATH)
print(f'loaded {len(bbo):,} BBO records')
print(f'instruments in file: {sorted(bbo.instrument_id.unique())}')

# Filter to a single instrument (first one in the file)
iid = bbo.instrument_id.iloc[0]
bbo = bbo[bbo.instrument_id == iid].reset_index(drop=True)
print(f'using instrument_id={iid}  rows={len(bbo):,}')

# Drop pathological rows (crossed / zero quotes)
before = len(bbo)
bbo = bbo[(bbo.bid > 0) & (bbo.ask > 0) & (bbo.ask > bbo.bid)].reset_index(drop=True)
print(f'dropped {before - len(bbo):,} crossed/zero rows; {len(bbo):,} remain')

bbo['mid'] = (bbo.bid + bbo.ask) * 0.5
bbo.head()

## Stream OFI per BBO tick

`OFICalculator.update(bids, asks, timestamp_ns)` is the same method AS calls every tick — running it in Python for research means we measure exactly the OFI value the strategy would see.

In [ ]:
calc = bf.OFICalculator(OFI_CFG)

ofi_values = np.empty(len(bbo), dtype=float)
for i, row in enumerate(bbo.itertuples(index=False)):
    ofi_values[i] = calc.update(
        bids=[(row.bid, row.bid_qty)],
        asks=[(row.ask, row.ask_qty)],
        timestamp_ns=row.ts_ns,
    )
bbo['ofi'] = ofi_values
print(f'OFI computed: warm={calc.is_warm()}  range=[{ofi_values.min():.3f}, {ofi_values.max():.3f}]  mean={ofi_values.mean():.4f}')
bbo[['ts_ns', 'bid', 'ask', 'mid', 'ofi']].head()

## Forward returns at each horizon

For each row at time `t`, find the row whose `ts_ns >= t + horizon` (or skip if none), then compute `log(mid_fwd / mid)`. `pd.merge_asof` with `direction='forward'` is the idiomatic way.

In [ ]:
for name, horizon_ns in HORIZONS_NS.items():
    target_ts = bbo.ts_ns + horizon_ns
    # For each row, find the first BBO at-or-after target_ts.
    fwd = pd.merge_asof(
        pd.DataFrame({'target_ts': target_ts}),
        bbo[['ts_ns', 'mid']].rename(columns={'ts_ns': 'fwd_ts', 'mid': 'fwd_mid'}),
        left_on='target_ts',
        right_on='fwd_ts',
        direction='forward',
    )
    # Log returns. NaN where no forward row exists (end of file).
    bbo[f'ret_{name}'] = np.log(fwd.fwd_mid.values / bbo.mid.values)

bbo[['ts_ns', 'mid', 'ofi'] + [f'ret_{h}' for h in HORIZONS_NS]].head(10)

## IC per horizon

Spearman rank correlation is the standard IC formulation in quant research — robust to OFI outliers and any non-linear (but monotone) signal-return relationship. Pearson would assume linearity; we don't need that here.

In [ ]:
results = []
for name in HORIZONS_NS:
    col = f'ret_{name}'
    mask = bbo[col].notna() & bbo.ofi.notna()
    n = int(mask.sum())
    if n < 100:
        results.append({'horizon': name, 'n': n, 'IC': float('nan'), 'p_value': float('nan')})
        continue
    rho, pval = spearmanr(bbo.loc[mask, 'ofi'], bbo.loc[mask, col])
    results.append({'horizon': name, 'n': n, 'IC': rho, 'p_value': pval})

ic_df = pd.DataFrame(results)
ic_df

## Quick plot — OFI vs forward return scatter at the most-significant horizon

In [ ]:
import matplotlib.pyplot as plt

best = ic_df.loc[ic_df.IC.abs().idxmax()]
col = f'ret_{best.horizon}'
mask = bbo[col].notna() & bbo.ofi.notna()
x = bbo.loc[mask, 'ofi'].values
y = bbo.loc[mask, col].values

# Bin OFI into deciles, show mean forward return per decile.
deciles = pd.qcut(x, 10, duplicates='drop')
mean_y = pd.DataFrame({'ofi': x, 'ret': y}).groupby(deciles, observed=False)['ret'].mean()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(mean_y)), mean_y.values * 1e4)  # bps
ax.axhline(0, color='k', linewidth=0.5)
ax.set_xlabel(f'OFI decile (low → high)')
ax.set_ylabel(f'mean forward {best.horizon} return (bps)')
ax.set_title(f'OFI predictiveness — IC={best.IC:.4f}  N={int(best.n):,}')
plt.tight_layout()
plt.show()

## What to do with the result

- **IC > 0.02 at some horizon**: OFI has real edge at that horizon. Next steps: combine with other features (multi-feature regression), then walk-forward backtest sized to your fill assumptions.
- **IC < 0.01 at every horizon**: OFI alone isn't predictive on this data. Three follow-ups before giving up:
  1. Re-run on a different day / different instrument — predictiveness varies by regime.
  2. Try a different feature (microprice deviation, queue imbalance, vol-z-score).
  3. Conditional analysis: is OFI predictive only when |OFI| > some threshold? Bin and re-compute IC per OFI quantile.
- **IC sign flip across horizons**: short-horizon momentum, longer-horizon mean reversion. Common — exploit only the horizon you can actually trade at given your latency.

## Multi-feature IC comparison

OFI is one feature. The real research question is *which* feature predicts best at *which* horizon — different features carry information at different time scales (book-pressure for ms-scale, drift for s-scale, vol-state for tens of seconds).

Below: same canon data, same forward returns, IC computed across four features. Same Python wrappers, same C++ AS uses on the tick path — so the IC numbers are exactly the predictive power AS sees in production.

In [ ]:
# All four features are computed by streaming through the same bbo DataFrame.
# Halflives match AS defaults (vol=60s, drift=30s).

# OFI already in bbo from the streaming loop above; copy under the comparison name.
bbo['feat_ofi'] = bbo['ofi']

# σ² — per-second variance EWMA
bbo['feat_sigma2'] = bf.ewma_variance(bbo, halflife_s=60.0)

# µ — per-√s drift EWMA
bbo['feat_drift'] = bf.ewma_drift(bbo, halflife_s=30.0)

# Microprice deviation: (microprice - mid). Positive = buy pressure visible.
# Streaming through FairValueEstimator with Mode::kMicro to match AS exactly.
fv_cfg = bf.FairValueEstimator.Config()
fv_cfg.mode = bf.FairValueEstimator.Mode.Micro
fve = bf.FairValueEstimator(fv_cfg)
micro = [fve.estimate(bid_px=r.bid, ask_px=r.ask, bid_qty=r.bid_qty, ask_qty=r.ask_qty)
         for r in bbo.itertuples(index=False)]
bbo['feat_micro_dev'] = pd.Series(micro, index=bbo.index) - bbo['mid']

FEATURES = ['feat_ofi', 'feat_sigma2', 'feat_drift', 'feat_micro_dev']
bbo[FEATURES].describe()

## IC matrix — features × horizons

In [ ]:
ic_matrix = pd.DataFrame(index=FEATURES, columns=list(HORIZONS_NS.keys()), dtype=float)
n_matrix = pd.DataFrame(index=FEATURES, columns=list(HORIZONS_NS.keys()), dtype=int)

for feat in FEATURES:
    for h_name in HORIZONS_NS:
        ret_col = f'ret_{h_name}'
        mask = bbo[feat].notna() & bbo[ret_col].notna()
        n = int(mask.sum())
        if n < 100:
            ic_matrix.loc[feat, h_name] = float('nan')
        else:
            rho, _ = spearmanr(bbo.loc[mask, feat], bbo.loc[mask, ret_col])
            ic_matrix.loc[feat, h_name] = rho
        n_matrix.loc[feat, h_name] = n

print('IC matrix (Spearman):')
print(ic_matrix.round(4))
print('\nSample sizes:')
print(n_matrix)

## Heatmap — visual IC comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
vmax = max(0.05, ic_matrix.abs().max().max())  # symmetric colour around 0
im = ax.imshow(ic_matrix.values.astype(float), aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
ax.set_xticks(range(len(HORIZONS_NS)))
ax.set_xticklabels(list(HORIZONS_NS.keys()))
ax.set_yticks(range(len(FEATURES)))
ax.set_yticklabels(FEATURES)
ax.set_xlabel('forward-return horizon')
ax.set_ylabel('feature')
for i, feat in enumerate(FEATURES):
    for j, h in enumerate(HORIZONS_NS):
        v = ic_matrix.loc[feat, h]
        ax.text(j, i, f'{v:+.3f}', ha='center', va='center',
                color='white' if abs(v) > vmax * 0.5 else 'black', fontsize=10)
fig.colorbar(im, ax=ax, label='Spearman IC')
ax.set_title(f'Feature × horizon IC  (N≈{n_matrix.values.mean():,.0f} per cell)')
plt.tight_layout()
plt.show()

## How to read the heatmap

- **Brightest cell (any colour)** = best feature × horizon pair to start strategy work from. Sign tells you direction (positive IC = feature value rises before returns rise).
- **A feature's row reading across horizons** tells you the time scale at which it predicts. OFI typically lives in the 100ms-1s zone; drift lives further out.
- **A horizon's column** tells you which features the market gives you signal on at that horizon. If every cell in a column is near zero, that horizon is just noise.
- **Sign flips along a row** = short-horizon momentum, longer-horizon reversion (or vice versa). Common — exploit only the horizon you can actually trade at.
- **All cells near zero**: the canon day is genuinely featureless (rare), or there's a data bug (more likely — re-check the forward-return alignment cell).